# Bayesian Playlist Recommender — Project R.E.M.

**Doel**: Gegeven de fysiologische toestand van een deelnemer, welk afspeellijsttype (Calm / Neutral / Energy) geeft de beste stemmingsverbetering?

**Model**: Hiërarchisch Bayesiaans regressiemodel met partiële pooling over deelnemers. Deelnemers met weinig sessies lenen statistische kracht van de groep.

**Stemmingsscore**: Valentie-gewogen composiet. Negatieve emoties (gestresseerd, moe) krijgen een negatief teken:
- `composiet = valentie(emotie) × intensiteit`
- `mood_delta = composiet_na − composiet_voor`

Voorbeeld: "Gestresseerd (8)" → -8, daarna "Rustig (5)" → +5, delta = **+13** (grote verbetering).

---

### Voordat je dit notebook uitvoert

```bash
python scripts/analysis/bayesian_recommender.py --participants peer bosbes
```

Dit genereert de trace en aanbevelingen in `data/analysis/bayesian_recommender/`.

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import arviz as az

# Add analysis module to path
sys.path.insert(0, str(Path().resolve().parent / 'scripts' / 'analysis'))
sys.path.insert(0, str(Path().resolve().parent / 'scripts' / 'wearables'))

from bayesian_recommender import (
    build_model_data, load_checkin_sessions, recommend_playlist,
    build_hierarchical_model, composite_mood, emotion_valence,
    PLAYLIST_NAMES, PLAYLIST_MAP, VALENCE_MAP,
)

warnings.filterwarnings('ignore', message='Glyph .* missing from font')
warnings.filterwarnings('ignore', category=FutureWarning)

DATA_ROOT     = Path().resolve().parent / 'data'
ANALYSIS_ROOT = DATA_ROOT / 'analysis'
BAYES_ROOT    = ANALYSIS_ROOT / 'bayesian_recommender'

# ── Okabe-Ito palette ─────────────────────────────────────────────────────────
OI = {
    'black':          '#000000',
    'orange':         '#E69F00',
    'sky_blue':       '#56B4E9',
    'bluish_green':   '#009E73',
    'yellow':         '#F0E442',
    'blue':           '#0072B2',
    'vermilion':      '#D55E00',
    'reddish_purple': '#CC79A7',
}

PARTICIPANTS = {
    'bosbes':      {'color': OI['sky_blue'],       'emoji': '🫐'},
    'kokosnoot':   {'color': OI['orange'],          'emoji': '🥥'},
    'limoen':      {'color': OI['bluish_green'],    'emoji': '🍋‍🟩'},
    'peer':        {'color': OI['reddish_purple'],  'emoji': '🍐'},
    'kiwi':        {'color': OI['vermilion'],       'emoji': '🥝'},
    'watermeloen': {'color': OI['blue'],            'emoji': '🍉'},
}

PLAYLIST_COLORS = {
    'Calm':    '#4A90D9',
    'Neutral': '#7B7B7B',
    'Energy':  '#E8913A',
}

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0f1218', 'axes.facecolor': '#181e2a',
    'axes.edgecolor': '#232b3a', 'axes.labelcolor': '#c9d1d9',
    'axes.grid': True, 'grid.color': '#232b3a', 'grid.linewidth': 0.5,
    'text.color': '#c9d1d9', 'xtick.color': '#586475', 'ytick.color': '#586475',
    'legend.facecolor': '#181e2a', 'legend.edgecolor': '#232b3a',
    'font.family': 'monospace', 'font.size': 9, 'figure.dpi': 120,
})

---

## Data laden

We laden drie bronnen:
1. **MCMC trace** (`trace.nc`) — de volledige posterior van het Bayesiaans model
2. **Aanbevelingen** (`recommendations.json`, `all_recommendations.json`) — samengevatte posteriors per deelnemer
3. **Sessie-data** — ruwe check-in en biometrische data, opnieuw opgebouwd via `build_model_data()`

In [ ]:
# ── Load trace and recommendations ────────────────────────────────────────────
trace = az.from_netcdf(str(BAYES_ROOT / 'trace.nc'))

with open(BAYES_ROOT / 'recommendations.json') as f:
    recs_summary = json.load(f)

with open(BAYES_ROOT / 'all_recommendations.json') as f:
    recs_full = json.load(f)

param_summary = pd.read_csv(BAYES_ROOT / 'parameter_summary.csv', index_col=0)

# ── Rebuild model data ────────────────────────────────────────────────────────
# Auto-detect participants with biometrics
bio_participants = []
wearables_dir = DATA_ROOT / 'wearables'
if wearables_dir.exists():
    for d in sorted(wearables_dir.iterdir()):
        if (d / 'processed' / 'session_biometrics.csv').exists():
            bio_participants.append(d.name)

data, participant_codes = build_model_data(bio_participants)

# ── Also load raw check-in data for emotion detail ───────────────────────────
checkin_df = load_checkin_sessions()

print(f"Trace geladen: {trace.posterior.dims['chain']} chains × {trace.posterior.dims['draw']} draws")
print(f"Deelnemers: {participant_codes}")
print(f"Deelnemers met biometrie: {bio_participants}")
print(f"Totaal sessies: {len(data)} ({data['has_biometrics'].sum()} met biometrie, {(~data['has_biometrics']).sum()} check-in only)")

---

## Convergentiediagnostiek

Controleer of de MCMC-sampler goed heeft geconvergeerd:
- **R-hat < 1.01**: alle chains zijn het eens over dezelfde verdeling
- **ESS > 400**: genoeg effectieve samples voor betrouwbare schattingen
- **Geen divergenties**: de sampler is niet vast komen te zitten

In [ ]:
# ── Convergence summary ───────────────────────────────────────────────────────
rhat_ok = (param_summary['r_hat'] < 1.01).all()
ess_ok = (param_summary['ess_bulk'] > 400).all()

print(f"R-hat < 1.01:  {'PASS' if rhat_ok else 'FAIL'}")
print(f"ESS > 400:     {'PASS' if ess_ok else 'FAIL'}")
print(f"Divergences:   {int(trace.sample_stats['diverging'].sum().values)}")
print()

# Show key parameters
key_params = [p for p in param_summary.index
              if any(k in p for k in ['mu_alpha', 'mu_playlist', 'sigma', 'beta_stress', 'beta_bb', 'beta_hr', 'beta_hour'])
              and 'offset' not in p and '[' not in p]
print(param_summary.loc[key_params][['mean', 'sd', 'hdi_5.5%', 'hdi_94.5%', 'r_hat', 'ess_bulk']].round(2))

In [ ]:
# ── Trace plots for key parameters ────────────────────────────────────────────
az.plot_trace(
    trace,
    var_names=['mu_playlist', 'sigma', 'beta_stress', 'beta_bb'],
    compact=True,
    figsize=(14, 8),
)
plt.suptitle('Trace plots — kernparameters', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---

## Valentie-gewogen stemmingsdata

Overzicht van alle 29 sessies met de valentie-gewogen composietscore. Negatieve emoties (gestresseerd, moe) krijgen een negatief teken, positieve emoties (gemotiveerd, happy, rustig) een positief teken. Neutraal = 0 ongeacht intensiteit.

| Valentie | Emoties |
|----------|---------|
| **-1** | Gestresseerd of gespannen, Moe of ongemotiveerd, Moe en gespannen |
| **0** | Neutraal, Neutraal tot een goed gevoel |
| **+1** | Rustig, Gemotiveerd, Happy, Goed gevoel |

In [ ]:
# ── Mood data overview table ──────────────────────────────────────────────────
display_cols = ['participant', 'playlist', 'emotion_before', 'intensity_before',
                'mood_before_composite', 'emotion_after', 'intensity_after',
                'mood_after_composite', 'mood_delta']
print(checkin_df[display_cols].to_string(index=False))
print(f"\nTotaal: {len(checkin_df)} sessies, {checkin_df['participant'].nunique()} deelnemers")

In [ ]:
# ── Mean mood_delta by playlist type ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

playlist_stats = data.groupby('playlist')['mood_delta'].agg(['mean', 'std', 'count'])
playlist_stats = playlist_stats.reindex(PLAYLIST_NAMES)

bars = ax.bar(
    playlist_stats.index,
    playlist_stats['mean'],
    yerr=playlist_stats['std'],
    capsize=5,
    color=[PLAYLIST_COLORS[p] for p in playlist_stats.index],
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85,
)

# Annotate with n
for i, (pl, row) in enumerate(playlist_stats.iterrows()):
    ax.text(i, row['mean'] + row['std'] + 0.3, f"n={int(row['count'])}",
            ha='center', fontsize=9, color='#c9d1d9')

ax.axhline(0, color='#586475', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mood delta (valentie-gewogen)')
ax.set_title('Gemiddelde stemmingsverandering per afspeellijsttype (ruwe data)')
plt.tight_layout()
plt.show()

---

## Posterior verdelingen per deelnemer

Voor elke deelnemer tonen we de posterior verdeling van de verwachte stemmingsverbetering per afspeellijsttype. Dit is het kernresultaat van het Bayesiaans model.

**Lezen**: de x-as is de verwachte mood_delta. Het zwarte verticale streepje is het posterieure gemiddelde. De stippellijnen zijn de 89% geloofwaardigheidsintervallen. De rode stippellijn bij 0 = geen effect.

In [ ]:
# ── Per-participant posterior panels (2×3 grid) ──────────────────────────────
n_participants = len(participant_codes)
n_cols = 3
n_rows = (n_participants + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows), squeeze=False)
fig.suptitle('Posterior mood delta per deelnemer × afspeellijst', fontsize=14, y=1.01)

posterior = trace.posterior

for idx, participant in enumerate(participant_codes):
    row, col = divmod(idx, n_cols)
    ax_row = axes[row]

    p_idx = idx
    emoji = PARTICIPANTS.get(participant, {}).get('emoji', '')

    for k, (name, color) in enumerate(PLAYLIST_COLORS.items()):
        ax = ax_row[col] if k == 0 else ax_row[col]  # all on same axes

        alpha_samples = posterior['alpha'].values[:, :, p_idx].flatten()
        bp_samples = posterior['beta_playlist'].values[:, :, p_idx, k].flatten()
        predicted = alpha_samples + bp_samples

        mean_val = np.mean(predicted)
        ci_low, ci_high = np.percentile(predicted, [5.5, 94.5])

        ax_row[col].hist(predicted, bins=40, density=True, alpha=0.55,
                         color=color, edgecolor='none', label=f'{name} ({mean_val:+.1f})')
        ax_row[col].axvline(mean_val, color=color, linewidth=1.5, linestyle='-', alpha=0.8)

    ax_row[col].axvline(0, color='#D55E00', linewidth=0.8, linestyle=':', alpha=0.6)
    ax_row[col].set_title(f'{emoji} {participant}', fontsize=11)
    ax_row[col].set_xlabel('Verwachte mood delta')
    ax_row[col].legend(fontsize=8, loc='upper left')
    if col == 0:
        ax_row[col].set_ylabel('Dichtheid')

# Hide empty axes
for idx in range(n_participants, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    axes[row][col].set_visible(False)

plt.tight_layout()
plt.savefig(BAYES_ROOT / 'posterior_grid.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Groepsniveau afspeellijsteffecten

Het hiërarchische model schat een **groepsniveau** effect per afspeellijsttype (`mu_playlist`). Dit is het gemiddelde effect over alle deelnemers. Forest plot toont het 89% HDI (highest density interval).

In [ ]:
# ── Group-level playlist effects — forest plot ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Forest plot
ax = axes[0]
mu_pl = posterior['mu_playlist'].values  # (chains, draws, 3)
for k, name in enumerate(PLAYLIST_NAMES):
    samples = mu_pl[:, :, k].flatten()
    mean_val = np.mean(samples)
    ci_low, ci_high = np.percentile(samples, [5.5, 94.5])

    ax.plot([ci_low, ci_high], [k, k], color=PLAYLIST_COLORS[name], linewidth=3, alpha=0.7)
    ax.plot(mean_val, k, 'o', color=PLAYLIST_COLORS[name], markersize=10, zorder=5)
    ax.text(ci_high + 0.3, k, f'{mean_val:+.1f} [{ci_low:+.1f}, {ci_high:+.1f}]',
            va='center', fontsize=9, color='#c9d1d9')

ax.axvline(0, color='#D55E00', linewidth=0.8, linestyle=':', alpha=0.6)
ax.set_yticks(range(len(PLAYLIST_NAMES)))
ax.set_yticklabels(PLAYLIST_NAMES)
ax.set_xlabel('Groepsniveau mood delta (mu_playlist)')
ax.set_title('Forest plot — groepseffect per afspeellijst (89% HDI)')

# Bar chart comparison
ax = axes[1]
means = [np.mean(mu_pl[:, :, k].flatten()) for k in range(3)]
ci_lows = [np.percentile(mu_pl[:, :, k].flatten(), 5.5) for k in range(3)]
ci_highs = [np.percentile(mu_pl[:, :, k].flatten(), 94.5) for k in range(3)]
yerr_low = [m - lo for m, lo in zip(means, ci_lows)]
yerr_high = [hi - m for m, hi in zip(means, ci_highs)]

ax.bar(PLAYLIST_NAMES, means, yerr=[yerr_low, yerr_high], capsize=5,
       color=[PLAYLIST_COLORS[p] for p in PLAYLIST_NAMES],
       edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axhline(0, color='#586475', linewidth=0.8, linestyle='--')
ax.set_ylabel('mu_playlist (posterior mean)')
ax.set_title('Groepsgemiddelde afspeellijsteffect')

plt.tight_layout()
plt.savefig(BAYES_ROOT / 'group_effects.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Partiële pooling: shrinkage-effect

Het hiërarchische model trekt deelnemers met weinig sessies richting het groepsgemiddelde. Dit is het *shrinkage-effect*. De scatterplot vergelijkt het ruwe per-deelnemer gemiddelde (x) met de posterieure schatting (y).

- Punten op de diagonaal = geen shrinkage (veel data, model vertrouwt de individuele schatting)
- Punten dichter bij de horizontale stippellijn = sterke shrinkage (weinig data, model leunt op de groep)

In [ ]:
# ── Shrinkage plot: raw mean vs posterior mean ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))

raw_means = data.groupby('participant')['mood_delta'].mean()
group_mean = data['mood_delta'].mean()

for p_idx, participant in enumerate(participant_codes):
    # Posterior mean (alpha + mean of beta_playlist across playlist types)
    alpha_mean = float(posterior['alpha'].values[:, :, p_idx].mean())
    bp_mean = float(posterior['beta_playlist'].values[:, :, p_idx, :].mean())
    posterior_mean = alpha_mean + bp_mean

    raw_m = raw_means.get(participant, 0)
    n_sessions = len(data[data['participant'] == participant])
    color = PARTICIPANTS.get(participant, {}).get('color', '#888')
    emoji = PARTICIPANTS.get(participant, {}).get('emoji', '')

    ax.scatter(raw_m, posterior_mean, color=color, s=80 + 10 * n_sessions, zorder=5, edgecolors='white', linewidth=0.5)
    ax.annotate(f'{emoji} {participant} (n={n_sessions})', (raw_m, posterior_mean),
                textcoords='offset points', xytext=(8, 4), fontsize=8, color='#c9d1d9')

# Diagonal (no shrinkage)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, '--', color='#586475', linewidth=0.8, alpha=0.6, label='Geen shrinkage')

# Group mean
ax.axhline(group_mean, color='#D55E00', linewidth=0.8, linestyle=':', alpha=0.5, label=f'Groepsgemiddelde ({group_mean:.1f})')

ax.set_xlabel('Ruwe mood_delta (per-deelnemer gemiddelde)')
ax.set_ylabel('Posterior mood_delta (hiërarchisch model)')
ax.set_title('Shrinkage: partiële pooling trekt schaarse deelnemers naar de groep')
ax.legend(fontsize=8)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(BAYES_ROOT / 'shrinkage.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Biometrische coëfficiënten

Deze coëfficiënten gelden alleen voor deelnemers met smartwatch-data (peer, bosbes). Ze tonen welke fysiologische inputvariabelen de stemmingsuitkomst het meest beïnvloeden.

- **beta_stress**: effect van pre-sessie stress (hoger = gestresseerder voor de sessie)
- **beta_bb**: effect van Body Battery bij aanvang
- **beta_hr**: effect van hartfrequentie voor de sessie
- **beta_hour**: effect van het tijdstip van de dag

In [ ]:
# ── Biometric coefficients — forest plot ──────────────────────────────────────
bio_vars = ['beta_stress', 'beta_bb', 'beta_hr', 'beta_hour']
bio_labels = ['Pre-stress', 'Body Battery', 'Hartfrequentie', 'Uur van de dag']
bio_colors = [OI['vermilion'], OI['bluish_green'], OI['reddish_purple'], OI['orange']]

fig, ax = plt.subplots(figsize=(10, 4))

for i, (var, label, color) in enumerate(zip(bio_vars, bio_labels, bio_colors)):
    samples = posterior[var].values.flatten()
    mean_val = np.mean(samples)
    ci_low, ci_high = np.percentile(samples, [5.5, 94.5])

    ax.plot([ci_low, ci_high], [i, i], color=color, linewidth=3, alpha=0.7)
    ax.plot(mean_val, i, 'o', color=color, markersize=10, zorder=5)
    ax.text(ci_high + 0.1, i, f'{mean_val:+.2f} [{ci_low:+.2f}, {ci_high:+.2f}]',
            va='center', fontsize=9, color='#c9d1d9')

ax.axvline(0, color='#D55E00', linewidth=0.8, linestyle=':', alpha=0.6, label='Geen effect')
ax.set_yticks(range(len(bio_labels)))
ax.set_yticklabels(bio_labels)
ax.set_xlabel('Coëfficiënt (gestandaardiseerd)')
ax.set_title('Biometrische coëfficiënten — 89% HDI')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(BAYES_ROOT / 'biometric_coefficients.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Aanbevelingsoverzicht

Per deelnemer: welke afspeellijst wordt aanbevolen, met welke zekerheid?

De horizontale balkgrafiek toont de verwachte mood_delta per afspeellijst per deelnemer, met 89% geloofwaardigheidsintervallen als whiskers.

In [ ]:
# ── Recommendation summary table ─────────────────────────────────────────────
print(f"{'Deelnemer':<15} {'Aanbeveling':<10} {'Zekerheid':>10} {'Onzeker?':>10}")
print("─" * 50)
for p in participant_codes:
    rec = recs_full[p]
    emoji = PARTICIPANTS.get(p, {}).get('emoji', '')
    print(f"{emoji} {p:<13} {rec['recommended_playlist']:<10} {rec['confidence']:>9.0%} {'ja' if rec['uncertain'] else 'nee':>10}")

In [ ]:
# ── Horizontal bar: expected mood_delta per playlist per participant ──────────
fig, ax = plt.subplots(figsize=(12, 6))

y_positions = []
y_labels = []
y_pos = 0
bar_height = 0.25

for p_idx, participant in enumerate(participant_codes):
    emoji = PARTICIPANTS.get(participant, {}).get('emoji', '')
    for k, name in enumerate(PLAYLIST_NAMES):
        preds = recs_summary[participant][name]
        mean_val = preds['mean']
        ci_low = preds['ci_low']
        ci_high = preds['ci_high']

        y = y_pos + k * bar_height
        ax.barh(y, mean_val, height=bar_height * 0.9,
                xerr=[[mean_val - ci_low], [ci_high - mean_val]],
                capsize=3, color=PLAYLIST_COLORS[name], alpha=0.8,
                edgecolor='white', linewidth=0.3)

    y_labels.append(f'{emoji} {participant}')
    y_positions.append(y_pos + bar_height)
    y_pos += 1.2

ax.axvline(0, color='#D55E00', linewidth=0.8, linestyle=':', alpha=0.6)
ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel('Verwachte mood delta (89% CI)')
ax.set_title('Afspeellijstaanbevelingen per deelnemer')

# Legend
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=PLAYLIST_COLORS[n], label=n) for n in PLAYLIST_NAMES]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(BAYES_ROOT / 'recommendations_bar.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Interactieve aanbeveling: demo

Hieronder simuleren we een aanbeveling voor **peer** met een specifieke fysiologische toestand. Pas de waarden aan om te zien hoe de aanbeveling verschuift.

We tonen ook hoe het veranderen van het stressniveau de voorspelling beïnvloedt.

In [ ]:
# ── Rebuild model for recommend_playlist() ────────────────────────────────────
# We need the model object (with _participant_codes and _z_params) to call recommend_playlist()
model = build_hierarchical_model(data, participant_codes)

# ── Demo: recommend for peer with specific state ─────────────────────────────
demo_participant = 'peer'
rec = recommend_playlist(
    trace, model, demo_participant,
    pre_stress=55.0,   # moderately stressed
    bb_start=40,       # moderate body battery
    pre_hr=80.0,       # normal heart rate
    hour_of_day=8,     # morning
)

print(f"Deelnemer: {demo_participant}")
print(f"Input: stress=55, BB=40, HR=80, uur=8")
print()
print(f"{'Afspeellijst':<12} {'Gemiddelde':>10} {'89% CI':>20}")
print("─" * 45)
for pl, vals in rec['predictions'].items():
    print(f"{pl:<12} {vals['mean_delta']:>+10.2f} [{vals['ci_low']:>+6.2f}, {vals['ci_high']:>+6.2f}]")
print()
print(f"Aanbeveling: {rec['recommendation']}")

In [ ]:
# ── Sensitivity: how does pre_stress change the recommendation? ───────────────
fig, ax = plt.subplots(figsize=(10, 5))

stress_range = np.arange(20, 85, 5)

for name, color in PLAYLIST_COLORS.items():
    means = []
    ci_lows = []
    ci_highs = []
    for stress_val in stress_range:
        rec = recommend_playlist(
            trace, model, 'peer',
            pre_stress=float(stress_val), bb_start=40, pre_hr=80, hour_of_day=8,
        )
        means.append(rec['predictions'][name]['mean_delta'])
        ci_lows.append(rec['predictions'][name]['ci_low'])
        ci_highs.append(rec['predictions'][name]['ci_high'])

    ax.plot(stress_range, means, color=color, linewidth=2, label=name)
    ax.fill_between(stress_range, ci_lows, ci_highs, color=color, alpha=0.15)

ax.axhline(0, color='#586475', linewidth=0.8, linestyle='--')
ax.set_xlabel('Pre-sessie stressniveau')
ax.set_ylabel('Verwachte mood delta')
ax.set_title('Peer: hoe verandert de aanbeveling bij verschillende stressniveaus?')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(BAYES_ROOT / 'stress_sensitivity.png', bbox_inches='tight', dpi=150)
plt.show()

---

## Samenvatting

### Model
- **Type**: Hiërarchisch Bayesiaans regressiemodel met non-centered parameterisatie
- **Deelnemers**: 6 (bosbes, kiwi, kokosnoot, limoen, peer, watermeloen)
- **Sessies**: 29 totaal (10 met biometrie, 19 check-in only)
- **Convergentie**: R-hat < 1.01, ESS > 400, 0 divergenties

### Bevindingen
1. **Energy-afspeellijst** toont consistent de hoogste verwachte stemmingsverbetering over alle deelnemers
2. **Calm** scoort tweede, **Neutral** het laagst
3. De meeste aanbevelingen zijn **onzeker** (overlappende CI's) door de kleine steekproef
4. **Kokosnoot** heeft de meest zekere aanbeveling (12 sessies, 93% zekerheid)
5. Partiële pooling werkt: deelnemers met weinig sessies (kiwi: 2) worden richting de groep getrokken

### Beperkingen
- 29 sessies voor 6 deelnemers is onvoldoende voor sterke personalisatie
- Biometrische coëfficiënten zijn gebaseerd op slechts 10 sessies van 2 deelnemers
- Het model kan niet onderscheiden of Energy werkt door het type muziek of door confounds (tijdstip, pre-state)

### Export voor Streamlit
- `data/analysis/bayesian_recommender/recommendations.json` — direct bruikbaar
- `data/analysis/bayesian_recommender/trace.nc` — volledige posterior voor verdere analyse